# Agent development

- [Libs](#libs)
- [Settings](settings)
- [Base workflow](#base-workflow)
	- [Create llm object](#create-llm-object)
	- [Simple workflow graph](#simple-workflow-graph)
- [Base facts extractor](#base-facts-extractor)
	- [First simple extraction without schemas](#first-simple-extraction-without-schemas)
	- [First simple extraction with schemas](#first-simple-extraction-with-schemas)
- [Base tools calling](#base-tools-calling)
	- [Create task and get list of task](#create-task-and-get-list-of-tasks)

## Libs

In [6]:
import os
import getpass

In [193]:
import json
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, MessagesState, StateGraph
from pydantic import BaseModel, Field
from langchain.tools import tool
from enum import StrEnum

## Settings

In [ ]:
if os.environ["GIGACHAT_CREDENTIALS"] is None:
	os.environ["GIGACHAT_CREDENTIALS"] = getpass.getpass()

## Base workflow

### Create llm object

In [ ]:
llm = GigaChat(
    credentials=os.environ["GIGACHAT_CREDENTIALS"],
    scope=os.environ["GIGACHAT_SCOPE"],
    model=os.environ["GIGACHAT_MODEL"],
    verify_ssl_certs=False,
    timeout=120,
)

llm.invoke("ты тут?").to_json()

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'messages', 'AIMessage'],
 'kwargs': {'content': 'Да, я здесь. Готов общаться.',
  'response_metadata': {'token_usage': {'prompt_tokens': 19,
    'completion_tokens': 10,
    'total_tokens': 29,
    'precached_prompt_tokens': 3},
   'model_name': 'GigaChat-2-Max:2.0.30.01',
   'x_headers': {'x-request-id': 'd8897f33-19cf-495d-9a3a-21d139c2366a',
    'x-session-id': 'f0676ef4-3f79-4adf-9e3a-3d532054f036',
    'x-client-id': None},
   'finish_reason': 'stop'},
  'type': 'ai',
  'id': 'd8897f33-19cf-495d-9a3a-21d139c2366a',
  'usage_metadata': {'output_tokens': 10,
   'input_tokens': 19,
   'total_tokens': 29,
   'input_token_details': {'cache_read': 3}},
  'tool_calls': [],
  'invalid_tool_calls': []}}

### Simple workflow graph

In [67]:
system_template = """
Ты дружелюбный персональный помощник!
Отвечай как и веди беседу как человек. 
"""
system_msg = SystemMessage(system_template)

In [68]:
def call_llm(state: MessagesState):
	ai_message = llm.invoke([system_msg] + state['messages'])
	return {"messages":  state["messages"] + [ai_message]}

In [69]:
workflow = StateGraph(MessagesState)
workflow.add_node("call_llm", call_llm)
workflow.add_edge(START, "call_llm")
workflow.add_edge("call_llm", END)

graph = workflow.compile(checkpointer=MemorySaver())

In [ ]:
SEPARATOR = "-" * 100
STOP_COMMAND = "стоп"

def run_chat_loop(graph, thread_id: str = "test") -> None:
    """Запускает интерактивный цикл диалога с графом.

    Читает ввод пользователя из stdin, отправляет его в граф
    и печатает ответ AI. Завершается по команде STOP_COMMAND.
    """
    while True:
        user_input = input()
        if user_input == STOP_COMMAND:
            break

        state = graph.invoke(
            input={"messages": [HumanMessage(content=user_input)]},
            config={"configurable": {"thread_id": thread_id}},
        )

        *_, human_msg, ai_msg = state["messages"]

        print(SEPARATOR)
        print("[Human]:", human_msg.content)
        print("[AI]:", ai_msg.content.replace("\n\n","\n"))

#### First launch

In [ ]:
run_chat_loop(graph, thread_id="test")

----------------------------------------------------------------------------------------------------
[Human]: привет
[AI]: Приветик! Чем сегодня займёмся?
----------------------------------------------------------------------------------------------------
[Human]: давай кошек обсудим
[AI]: Отличная идея! Кошки – они же просто чудо какое-то. У тебя есть пушистый друг?
----------------------------------------------------------------------------------------------------
[Human]: да у меня есть миюки
[AI]: Какое красивое имя! Миюки, наверное, очень ласковая и игривая кошечка. Расскажи немного о ней, какая она?
----------------------------------------------------------------------------------------------------
[Human]: она самая лучшая
[AI]: Это точно! Все кошки особенные, но когда они наши, становятся самыми лучшими. А что Миюки любит больше всего делать? Играть, спать или, может, лакомства обожает?
--------------------------------------------------------------------------------------------

#### Second launch

In [71]:
run_chat_loop(graph, thread_id="test")

----------------------------------------------------------------------------------------------------
[Human]: привет!
[AI]: Привет-привет! Рад снова слышать тебя. Что нового?
----------------------------------------------------------------------------------------------------
[Human]: помнишь я тебе о кошке рассказывал
[AI]: Да-да, конечно помню! Ты говорил про свою замечательную кошечку Миюки. Как у неё дела?
----------------------------------------------------------------------------------------------------
[Human]: да у нее все хорошо
[AI]: Рада это слышать! Надеюсь, она продолжает радовать вас своей мурчательной компанией. Есть какие-нибудь забавные истории или свежие новости про неё?
----------------------------------------------------------------------------------------------------
[Human]: надо бежать пока
[AI]: Хорошо, береги себя! И передай привет Миюки, пусть тоже бережет свой пушистый хвостик. До скорой встречи!


## Base facts extractor

In [78]:
system_template = """
Ты профессиональный сборщик фактов о пользователе.
Твоя задача достать вычленять факты из сообщения пользователя.
"""
system_msg = SystemMessage(system_template)

#### First simple extraction without schemas

In [86]:
human_msg_1 = HumanMessage(
	content="Привет! Меня зовут Алексей, мне 34 года, я работаю backend-разработчиком в финтехе. Живу в Санкт-Петербурге."
)

ai_msg = llm.invoke([system_msg] + [human_msg_1])
print(ai_msg.content)

Факты о пользователе:
1. Имя пользователя – Алексей.
2. Возраст – 34 года.
3. Профессия – backend-разработчик.
4. Сфера деятельности – финтех.
5. Место проживания – Санкт-Петербург.


In [87]:
human_msg_2 = HumanMessage(
	content="У меня двое детей — сын Максим 8 лет и дочь София 3 года. Жена работает врачом. Мы переехали в Берлин два года назад."
)
ai_msg = llm.invoke([system_msg] + [human_msg_2])
print(ai_msg.content)

Факты из вашего сообщения:
1. У вас двое детей:
   - Сын Максим, возраст 8 лет.
   - Дочь София, возраст 3 года.
2. Ваша жена работает врачом.
3. Вы переехали в город Берлин два года назад.


In [89]:
human_msg_3 = HumanMessage(
	content="Вообще-то я не люблю спорт, но по выходным хожу в бассейн. Ещё увлекаюсь астрофотографией и коллекционирую виниловые пластинки."
)
ai_msg = llm.invoke([system_msg] + [human_msg_3])
print(ai_msg.content)

Факты о пользователе:
1. Не любит спорт в целом.
2. Ходит в бассейн по выходным.
3. Увлекается астрофотографией.
4. Коллекционирует виниловые пластинки.


#### First simple extraction with schemas

In [99]:
class Fact(BaseModel):
    text: str = Field(description="Факт")         
    category: str = Field(description="Категория факта")  

class FactsExtraction(BaseModel):
    """Извлеченные факты о пользователе"""
    facts: list[Fact]

llm_facts_extractor = llm.with_structured_output(FactsExtraction)

In [103]:
facts: FactsExtraction = llm_facts_extractor.invoke([system_msg] + [human_msg_1])
for fact in facts.facts:
	print(fact)

text='Имя пользователя - Алексей' category='Имя'
text='Возраст пользователя - 34 года' category='Возраст'
text='Профессия пользователя - backend-разработчик в финтехе' category='Профессия'
text='Место жительства пользователя - Санкт-Петербург' category='Место жительства'


In [104]:
facts: FactsExtraction = llm_facts_extractor.invoke([system_msg] + [human_msg_2])
for fact in facts.facts:
	print(fact)

text='У пользователя двое детей: сын Максим 8 лет и дочь София 3 года.' category='Семья'
text='Жена пользователя работает врачом.' category='Семья'
text='Пользователь переехал в Берлин два года назад.' category='Место жительства'


In [105]:
facts: FactsExtraction = llm_facts_extractor.invoke([system_msg] + [human_msg_3])
for fact in facts.facts:
	print(fact)

text='Не любит спорт, но ходит в бассейн по выходным' category='Хобби и увлечения'
text='Увлекается астрофотографией' category='Хобби и увлечения'
text='Коллекционирует виниловые пластинки' category='Хобби и увлечения'


## Base tools calling

### Create task and get list of tasks

In [219]:
class TaskStatus(StrEnum):
    """Статусы задачи."""
    TODO = "TODO"
    IN_PROGRESS = "IN_PROGRESS"
    DONE = "DONE"
    CANCELLED = "CANCELLED"


class Task(BaseModel):
    """Модель задачи."""
    description: str = Field(description="Описание задачи")
    status: TaskStatus = Field(
        default=TaskStatus.TODO,
        description="Текущий статус задачи",
    )

_tasks: list[Task] = []

@tool
def create_task(task: str) -> Task:
    """Создать ОДНУ задачу.

    ВАЖНО: если пользователь перечисляет несколько задач,
    вызывай этот инструмент несколько раз — по одному вызову на задачу.
    """
    new_task = Task(description=task)
    _tasks.append(new_task)
    return new_task


@tool
def get_task_list() -> list[Task]:
    """Возвращает список всех задач."""
    return _tasks

available_tools_list : list = [create_task, get_task_list]
available_tools_dict: dict = {tool.name: tool for tool in available_tools_list}

In [220]:
llm_with_tools = llm.bind_tools(available_tools_list)

ai_msg.to_json()

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'messages', 'AIMessage'],
 'kwargs': {'content': '',
  'additional_kwargs': {'function_call': {'name': 'create_task',
    'arguments': {'task': 'Сделать домашнюю работу'}},
   'functions_state_id': '01a0b921-6ef8-7b99-81c2-0b973f6e3c63'},
  'response_metadata': {'token_usage': {'prompt_tokens': 84,
    'completion_tokens': 38,
    'total_tokens': 122,
    'precached_prompt_tokens': 5},
   'model_name': 'GigaChat-2-Max:2.0.30.01',
   'x_headers': {'x-request-id': '5dd0ece0-293e-4edd-ab8a-75fdafcafd79',
    'x-session-id': '670f5d56-52bd-4376-969c-013d9f036e87',
    'x-client-id': None},
   'finish_reason': 'function_call'},
  'type': 'ai',
  'id': '5dd0ece0-293e-4edd-ab8a-75fdafcafd79',
  'tool_calls': [{'name': 'create_task',
    'args': {'task': 'Сделать домашнюю работу'},
    'id': '53586a44-8c19-452a-acd4-83a860dfe695',
    'type': 'tool_call'}],
  'usage_metadata': {'output_tokens': 38,
   'input_tokens': 84,
   'tota

In [ ]:
def call_llm_with_tools(query: str, max_attempt: int = 10):

	messages: list = []
	human_msg = HumanMessage(content=query)
	messages.append(human_msg)

	ai_msg = llm_with_tools.invoke(messages)
	messages.append(ai_msg)


	for _ in range(max_attempt):
		if len(ai_msg.tool_calls) == 0:
			break
	
		for tool_call in ai_msg.tool_calls:
			name: str = tool_call["name"]
			args: dict = tool_call["args"]
			id: str = tool_call["id"]
	
			tool_result = available_tools_dict[name].invoke(args)
			tool_message = ToolMessage(
				content=json.dumps(tool_result, ensure_ascii=False, default=str),
				tool_call_id=id,
				name=name,
			)
			messages.append(tool_message)

			ai_msg = llm_with_tools.invoke(messages)
			messages.append(ai_msg)

	for msg in messages:
		if isinstance(msg, HumanMessage):
			print("[Human]:", msg.content)
		if isinstance(msg, AIMessage):
			print("[AI]:", msg.content)
		if isinstance(msg, ToolMessage):
			print("[tool]:", msg.content)

In [ ]:
_tasks = []

print("### Первый запуск, без создания задачи ###")
call_llm_with_tools("Привет")
print()
print("current_task_list:",_tasks)
print("-" * 50)

print("### Проверка что список задач пуст ###")
call_llm_with_tools("Какие у меня задачи на сегодня")
print()
print("current_task_list:", _tasks)
print("-" * 50)

print("### Создание первой задачи ###")
call_llm_with_tools("Мне нужно сделать домашнюю работу")
print()
print("current_task_list:", _tasks)
print("-" * 50)

print("### Получения списка задач (с одной задачей) ###")
call_llm_with_tools("Что у меня по задачам на сегодня?")
print()
print("current_task_list:", _tasks)
print("-" * 50)

print("### Создание двух дополнительных задач ###")
call_llm_with_tools("Мне нужно сегодня помыть посуду и выгулять собаку")
print()
print("current_task_list:", _tasks)
print("-" * 50)

print("### Проверка списка задач ###")
call_llm_with_tools("Какие у меня задачи на сегодня")
print()
print("current_task_list:", _tasks)
print("-" * 50)

### Первый запуск, без создания задачи ###
[Human]: Привет
[AI]: Привет!

current_task_list: []
--------------------------------------------------
### Проверка что список задач пуст ###
[Human]: Какие у меня задачи на сегодня
[AI]: 
[tool]: []
[AI]: На сегодня у вас пока нет запланированных задач.

current_task_list: []
--------------------------------------------------
### Создание первой задачи ###
[Human]: Мне нужно сделать домашнюю работу
[AI]: 
[tool]: "description='Сделать домашнюю работу' status=<TaskStatus.TODO: 'TODO'>"
[AI]: Задача успешно создана: **Сделать домашнюю работу**.

current_task_list: [Task(description='Сделать домашнюю работу', status=<TaskStatus.TODO: 'TODO'>)]
--------------------------------------------------
### Получения списка задач (с одной задачей) ###
[Human]: Что у меня по задачам на сегодня?
[AI]: 
[tool]: ["description='Сделать домашнюю работу' status=<TaskStatus.TODO: 'TODO'>"]
[AI]: Сегодняшние задачи:
- Сделать домашнюю работу

current_task_list: [